# Notebook 6 Style — Tiny GPT Literature Generator

이 노트북은 수업시간의 `notebook_06` 형태를 바탕으로, 로미오와 줄리엣 같은 문학 텍스트를 학습하는 작은 GPT를 만듭니다.

핵심 구성:

- character-level vocabulary
- NextTokenDataset
- multi-head masked self-attention
- feedforward network
- residual connection
- layer normalization
- block stacking
- sampling

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# data/input.txt를 원하는 문학 텍스트로 바꾸면 됩니다.
# 황순원의 「소나기」는 저작권이 남아 있을 수 있으므로 직접 합법적으로 확보한 텍스트만 사용하세요.
data_path = Path("data/input.txt")
text = data_path.read_text(encoding="utf-8")

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab size:", vocab_size)
print("sample chars:", ''.join(chars[:50]))

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64
batch_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

## 1. Multi-head masked self-attention

각 위치는 자기 자신과 이전 위치만 볼 수 있습니다. 미래 위치는 causal mask로 막습니다.

In [ ]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert emb_dim % num_heads == 0
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

## 2. Feedforward + Block

Attention 뒤에 MLP를 붙이고, residual connection과 layer normalization을 사용합니다.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 3. Tiny GPT

토큰 임베딩과 위치 임베딩을 더한 뒤, Transformer block 여러 개를 통과시킵니다.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

model = TinyGPT(vocab_size, block_size)
logits, loss = model(xb, yb)
print("logits.shape:", logits.shape)
print("loss:", loss.item())

## 4. 학습

In [ ]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=20):
    model.eval()
    losses = []
    for i, (xb, yb) in enumerate(loader):
        if i >= eval_iters:
            break
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

max_iters = 500
eval_interval = 100

for step in range(max_iters):
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0 or step == max_iters - 1:
        train_loss = estimate_loss(model, loader, device, eval_iters=10)
        print(f"step {step:4d} | train loss {train_loss:.4f}")

## 5. Sampling

학습된 모델은 지금까지 나온 문자들을 보고 다음 문자를 하나씩 뽑습니다.

In [ ]:
@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400, temperature=1.0):
    model.eval()
    encoded = [stoi[ch] for ch in start_text if ch in stoi]
    if len(encoded) == 0:
        encoded = [0]
    idx = torch.tensor([encoded], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, ix], dim=1)
    return "".join(itos[i] for i in idx[0].tolist())

print(sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=500))

## 6. 저장

스크립트 버전에서는 `python train.py`를 실행하면 자동으로 `checkpoints/tiny_gpt.pt`에 저장됩니다.

In [ ]:
Path("checkpoints").mkdir(exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_embd": 128,
        "n_head": 4,
        "n_layer": 4,
        "dropout": 0.1,
    },
    "stoi": stoi,
    "itos": itos,
}, "checkpoints/tiny_gpt.pt")
print("saved checkpoints/tiny_gpt.pt")

## 7. 정리

이 repository는 `notebook_06`에서 배운 작은 GPT 구조를 과제 제출용으로 정리한 것입니다.

GitHub에 올릴 때는 전체 폴더를 그대로 업로드하면 됩니다. 단, `checkpoints/*.pt`는 용량이 커질 수 있어서 `.gitignore`에 제외되어 있습니다.